# FedCircuitNet - seed sweep of the client-count runs (RouteNetGroupNorm)

Runs from `sweep_seed_config/` (`run_sweep_seed.sh`): `iid` and `FH` (feature_hierarchical) partitioning at K = 5 / 10 / 20 clients (full participation, `round_clients = n_partitions`, T = 1600 rounds), each repeated for seeds 0 / 1 / 2. The seed drives `runtime.seed` and the partitioner seed, so a seed changes the client split as well as the training randomness.

Every curve is the **point-wise mean over seeds** (band = +- 1 std, thin lines = individual seeds); every scalar is reported as mean +- std over seeds (sample std, ddof = 1). Only train and test are swept - there are no augmented metrics.

## Setup and helpers

In [ ]:
# Registers the PNG formatter up front; without it the first plotting cell
# renders its figures as text under the pinned matplotlib 3.7.
%matplotlib inline

import os
import re
import warnings
from collections import defaultdict
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from aim import Repo
from aim.storage.context import Context

# Same ROC / PR-AUC code test_aim.py runs (verbatim port of the CircuitNet reference).
from utils.metrics import calculate_all, get_sorted_list, roc_prc

repo = Repo(path=os.getcwd())

# Experiment names set in sweep_seed_config/*.yaml.
TRAIN_EXPERIMENT = 'fedavg_drc_train_sweep_seed'
TEST_EXPERIMENT = 'fedavg_drc_test_sweep_seed'

SCHEMES = ['iid', 'FH']

# Aim tag of both phases (`tag:` in the yaml), e.g. FH_40KL_c10_seed2.
RUN_ID = re.compile(r'^(?P<scheme>' + '|'.join(SCHEMES) + r')_40KL_c(?P<K>\d+)_seed(?P<seed>\d+)$')

SCHEME_COLOR = {'iid': '#2a78d6', 'FH': '#eda100'}
SCHEME_MARKER = {'iid': 'o', 'FH': 'D'}
K_RAMP = ['#86b6ef', '#5598e7', '#256abf', '#184f95', '#0d366b']
INK, MUTED = '#0b0b0b', '#898781'

plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#c3c2b7',
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'axes.axisbelow': True,
    'axes.titlesize': 10,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6,
    'lines.linewidth': 1.6,
    'legend.frameon': False,
    'figure.dpi': 110,
})

In [ ]:
def runs_of(experiment):
    query = (
        'run.active == False and run.archived == False '
        f'and run.experiment == "{experiment}"'
    )
    return [w.run for w in repo.query_runs(query=query, report_mode=0).iter_runs()]


def run_key(run):
    # (scheme, K, seed) from the run's Aim tag, None if it is not a sweep run.
    ids = [RUN_ID.match(str(t)) for t in run.tags]
    ids = [m for m in ids if m]
    return (ids[0]['scheme'], int(ids[0]['K']), int(ids[0]['seed'])) if len(ids) == 1 else None


def collect_runs(experiment, phase):
    # {(scheme, K, seed): run}; on re-runs the latest run wins.
    found, ignored = defaultdict(list), 0
    for run in runs_of(experiment):
        key = run_key(run)
        if key is None:
            ignored += 1
            continue
        found[key].append(run)
    out = {}
    for key, runs in found.items():
        runs.sort(key=lambda r: r.creation_time)
        if len(runs) > 1:
            print(f'[{phase}] {key}: {len(runs)} runs, keeping the latest ({runs[-1].hash})')
        out[key] = runs[-1]
    if ignored:
        print(f'[{phase}] ignored {ignored} run(s) in {experiment} outside the sweep')
    return out


def by_cell(runs):
    # {(scheme, K): {seed: run}}
    cells = defaultdict(dict)
    for (scheme, K, seed), run in runs.items():
        cells[(scheme, K)][seed] = run
    return cells


def metric_series(run, name, context=None):
    # (steps, values) of a tracked metric, sorted by step; empty if missing.
    metric = run.get_metric(name, context=Context(context or {}))
    if metric is None:
        return np.array([]), np.array([])
    steps, (vals,) = metric.data.view('val').numpy()
    order = np.argsort(steps, kind='stable')
    return steps[order], np.asarray(vals, dtype=float)[order]


def last_value(run, name, context=None):
    _, vals = metric_series(run, name, context)
    return float(vals[-1]) if vals.size else np.nan


def per_sample(run, name):
    return metric_series(run, name, {'subset': 'test', 'aggregation': 'distribution'})[1]


def mean_curve(series):
    # Point-wise mean / std over seeds of [(steps, values), ...], on the steps
    # every seed logged.  Returns (steps, mean, std, stacked) or None.
    series = [(s, v) for s, v in series if v.size]
    if not series:
        return None
    common = reduce(np.intersect1d, [s for s, _ in series])
    stacked = np.stack([v[np.isin(s, common)] for s, v in series])
    return common, stacked.mean(0), stacked.std(0, ddof=1) if len(series) > 1 else np.zeros(stacked.shape[1]), stacked


def mark_missing(ax, title, text='not run'):
    ax.set_title(title)
    ax.text(0.5, 0.5, text, ha='center', va='center', transform=ax.transAxes, color=MUTED)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def scheme_legend(ax, **kw):
    handles = [plt.Line2D([], [], color=SCHEME_COLOR[s], marker=SCHEME_MARKER[s], label=s) for s in SCHEMES]
    ax.legend(handles=handles, fontsize=8, **kw)


def log_k_axis(ax):
    ax.set_xscale('log')
    ax.set_xticks(KS)
    ax.set_xticklabels([str(k) for k in KS])
    ax.minorticks_off()
    ax.set_xlabel('K (clients)')


def plot_band(ax, x, mean, std, stacked, color, label, marker=None):
    for row in stacked:
        ax.plot(x, row, color=color, alpha=0.2, linewidth=0.6)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.2, linewidth=0)
    ax.plot(x, mean, color=color, label=label, marker=marker, markevery=max(1, len(x) // 10))

## Sweep inventory

In [ ]:
train_runs = collect_runs(TRAIN_EXPERIMENT, 'train')
test_runs = collect_runs(TEST_EXPERIMENT, 'test')
train_cells, test_cells = by_cell(train_runs), by_cell(test_runs)

# Save folders are reused across re-runs: a test run older than its train run
# scored whatever checkpoint the folder held before.
for key, run in test_runs.items():
    if key in train_runs and run.creation_time < train_runs[key].creation_time:
        print(f'[test] {key}: run {run.hash} predates its train run - stale checkpoint?')

KS = sorted({K for _, K, _ in list(train_runs) + list(test_runs)})
SEEDS = sorted({seed for _, _, seed in list(train_runs) + list(test_runs)})
K_COLOR = dict(zip(KS, K_RAMP if len(KS) <= len(K_RAMP)
                   else plt.get_cmap('Blues')(np.linspace(0.4, 0.95, len(KS)))))

inventory = pd.DataFrame([
    {'K': K, 'scheme': s,
     'train seeds': sorted(train_cells.get((s, K), {})) or '-',
     'test seeds': sorted(test_cells.get((s, K), {})) or '-'}
    for K in KS for s in SCHEMES
]).set_index(['K', 'scheme'])
if inventory.empty:
    print('No sweep runs found - run ./run_sweep_seed.sh first.')
else:
    display(inventory)
    expected = len(SEEDS)
    for (s, K), runs in {**train_cells, **test_cells}.items():
        if len(runs) != expected:
            print(f'WARNING: {s} K={K} has {len(runs)} of {expected} seeds')

## Training curves

Mean client train loss per round, averaged point-wise over seeds. Bold: mean over seeds, band: +- 1 std, thin: single seeds. `SMOOTH` is a centred rolling mean applied to the seed-mean for the summary table only.

In [ ]:
SMOOTH = 10

mean_losses = {}   # (scheme, K) -> mean_curve of 'Mean Client Train Loss'
for cell, runs in train_cells.items():
    mean_losses[cell] = mean_curve([metric_series(r, 'Mean Client Train Loss', {'subset': 'train'})
                                    for r in runs.values()])

if mean_losses:
    fig, axes = plt.subplots(1, len(KS), figsize=(4.6 * len(KS), 3.8), constrained_layout=True,
                             sharey=True, squeeze=False)
    for ax, K in zip(axes[0], KS):
        plotted = False
        for scheme in SCHEMES:
            curve = mean_losses.get((scheme, K))
            if curve is None:
                continue
            steps, mean, std, stacked = curve
            plot_band(ax, steps, mean, std, stacked, SCHEME_COLOR[scheme],
                      f'{scheme} ({len(stacked)} seeds)', SCHEME_MARKER[scheme])
            plotted = True
        if not plotted:
            mark_missing(ax, f'K = {K}')
            continue
        ax.set_title(f'K = {K}')
        ax.set_xlabel('round')
        ax.legend(fontsize=8)
    axes[0, 0].set_ylabel('mean client train loss')
    fig.suptitle('Mean client train loss, mean +- std over seeds')
    plt.show()
    plt.close(fig)

    # Cross-size comparison, one panel per scheme, one line per K.
    fig, axes = plt.subplots(1, len(SCHEMES), figsize=(4.6 * len(SCHEMES), 3.6), constrained_layout=True, sharey=True)
    for ax, scheme in zip(axes, SCHEMES):
        plotted = False
        for K in KS:
            curve = mean_losses.get((scheme, K))
            if curve is None:
                continue
            steps, mean, std, _ = curve
            ax.plot(steps, pd.Series(mean).rolling(SMOOTH, min_periods=1, center=True).mean(),
                    color=K_COLOR[K], label=f'K = {K}')
            plotted = True
        if not plotted:
            mark_missing(ax, scheme)
            continue
        ax.set_title(scheme)
        ax.set_xlabel('round')
        ax.legend(fontsize=8)
    axes[0].set_ylabel(f'seed-mean train loss (rolling {SMOOTH})')
    plt.show()
    plt.close(fig)

# Per-seed final loss (mean over the last SMOOTH rounds), aggregated over seeds.
train_final = {}
for (scheme, K), runs in train_cells.items():
    for seed, run in runs.items():
        _, vals = metric_series(run, 'Mean Client Train Loss', {'subset': 'train'})
        train_final[(scheme, K, seed)] = float(vals[-SMOOTH:].mean()) if vals.size else np.nan

## Test configurations

In [ ]:
rows = []
for (scheme, K, seed), run in sorted(test_runs.items(), key=lambda kv: (kv[0][1], kv[0][0], kv[0][2])):
    hparams = run.get('hparams') or {}
    evaluation = hparams.get('evaluation') or {}
    rows.append({
        'K': K, 'scheme': scheme, 'seed': seed,
        'checkpoint': hparams.get('checkpoint'),
        'threshold': evaluation.get('threshold'),
        'save_path': evaluation.get('save_path'),
        'run': run.hash,
    })
if rows:
    display(pd.DataFrame(rows).set_index(['K', 'scheme', 'seed']))

## ROC and PR curves

ROC-AUC and PR-AUC come from `roc_prc.csv` of each run's `evaluation.save_path` (read from the working directory, so run the notebook where the sweep wrote `work_dirs/`). The mean curve interpolates every seed on a common FPR (resp. recall) grid and averages point-wise.

In [ ]:
MAX_FPR = 0.10
FPR_GRID = np.unique(np.r_[np.linspace(0, MAX_FPR, 501), np.linspace(MAX_FPR, 1, 200)])
RECALL_GRID = np.linspace(0, 1, 501)


def roc_pr_curves(run):
    save_path = ((run.get('hparams') or {}).get('evaluation') or {}).get('save_path')
    csv = os.path.join(os.getcwd(), save_path or '', 'roc_prc.csv')
    if not save_path or not os.path.isfile(csv):
        print(f'[{run.hash}] no roc_prc.csv under {save_path} - ROC / PR skipped')
        return None
    roc_auc, pr_auc = roc_prc(save_path)
    tpr, fpr, precision = calculate_all(csv)
    fpr_s, tpr_s = get_sorted_list(fpr, tpr)
    recall_s, precision_s = get_sorted_list(tpr, precision)
    fpr_c, tpr_c = np.r_[fpr_s, 1.0], np.r_[tpr_s, 1.0]
    return {
        'roc_auc': float(roc_auc), 'pr_auc': float(pr_auc),
        'fpr': fpr_c, 'tpr': tpr_c,
        'recall': np.asarray(recall_s), 'precision': np.asarray(precision_s),
        'tpr@fpr5': float(np.interp(0.05, fpr_c, tpr_c)) if fpr_c[0] <= 0.05 else np.nan,
    }


curves = {key: roc_pr_curves(run) for key, run in test_runs.items()}


def seed_mean_curve(cell, kind):
    # Point-wise mean +- std over the seeds of a (scheme, K) cell.
    cs = [curves.get((cell[0], cell[1], seed)) for seed in sorted(test_cells.get(cell, {}))]
    cs = [c for c in cs if c]
    if not cs:
        return None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        if kind == 'roc':
            grid, stacked = FPR_GRID, np.stack([np.interp(FPR_GRID, c['fpr'], c['tpr']) for c in cs])
        else:
            grid = RECALL_GRID
            stacked = np.stack([np.interp(RECALL_GRID, c['recall'], c['precision'], left=np.nan, right=np.nan)
                                for c in cs])
        mean = np.nanmean(stacked, axis=0)
        std = np.nanstd(stacked, axis=0, ddof=1) if len(cs) > 1 else np.zeros_like(mean)
    return grid, mean, std, stacked, cs


def cell_auc(cs, key):
    vals = np.array([c[key] for c in cs])
    return vals.mean(), (vals.std(ddof=1) if len(vals) > 1 else 0.0)


def draw_curve(ax, cell, kind, color, label):
    out = seed_mean_curve(cell, kind)
    if out is None:
        return False
    grid, mean, std, stacked, cs = out
    m, s = cell_auc(cs, 'roc_auc' if kind == 'roc' else 'pr_auc')
    name = 'ROC-AUC' if kind == 'roc' else 'PR-AUC'
    if kind == 'roc':
        keep = grid <= MAX_FPR
        grid, mean, std, stacked = grid[keep], mean[keep], std[keep], stacked[:, keep]
    for row in stacked:
        ax.plot(grid, row, color=color, alpha=0.2, linewidth=0.6)
    ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.2, linewidth=0)
    ax.plot(grid, mean, color=color, label=f'{label} ({name}={m:.4f}+-{s:.4f})')
    return True


def finish(ax, kind, title):
    if kind == 'roc':
        ax.plot([0, MAX_FPR], [0, MAX_FPR], linestyle='--', color=MUTED, linewidth=1)
        ax.set_xlim(0, MAX_FPR)
        ax.set_xlabel('false positive rate')
        ax.set_ylabel('true positive rate')
    else:
        ax.set_xlim(0, 1)
        ax.set_xlabel('recall')
        ax.set_ylabel('precision')
    ax.set_ylim(0, 1)
    ax.set_title(title)
    if ax.lines:
        ax.legend(fontsize=7, loc='center right' if kind == 'roc' else 'upper right')


if any(curves.values()):
    for kind in ('roc', 'pr'):
        # One panel per K, schemes overlaid.
        fig, axes = plt.subplots(1, len(KS), figsize=(4.8 * len(KS), 4.2), constrained_layout=True, squeeze=False)
        for ax, K in zip(axes[0], KS):
            for scheme in SCHEMES:
                draw_curve(ax, (scheme, K), kind, SCHEME_COLOR[scheme], scheme)
            finish(ax, kind, f'K = {K}')
        fig.suptitle(f'{kind.upper()} curves, mean +- std over seeds')
        plt.show()
        plt.close(fig)
        # One panel per scheme, K overlaid.
        fig, axes = plt.subplots(1, len(SCHEMES), figsize=(4.8 * len(SCHEMES), 4.2), constrained_layout=True)
        for ax, scheme in zip(axes, SCHEMES):
            for K in KS:
                draw_curve(ax, (scheme, K), kind, K_COLOR[K], f'K = {K}')
            finish(ax, kind, scheme)
        plt.show()
        plt.close(fig)

## Test metric distributions

Per-sample NRMS / SSIM over the test set. The boxes pool the samples of all seeds of a cell (whiskers: 5th-95th percentile); the per-seed average is what enters the final table.

In [ ]:
EVAL_METRICS = ['NRMS', 'SSIM']
test_values = {key: {m: per_sample(run, f'Test {m} (dist)') for m in EVAL_METRICS}
               for key, run in test_runs.items()}


def pooled(cell, metric):
    vals = [test_values[(cell[0], cell[1], seed)][metric] for seed in sorted(test_cells.get(cell, {}))]
    vals = [v for v in vals if v.size]
    return np.concatenate(vals) if vals else None


def grouped_boxplot(ax, metric):
    width = 0.8 / len(SCHEMES)
    for i, scheme in enumerate(SCHEMES):
        pos, vals = [], []
        for g, K in enumerate(KS):
            v = pooled((scheme, K), metric)
            if v is not None:
                pos.append(g + (i - (len(SCHEMES) - 1) / 2) * width)
                vals.append(v)
        if not vals:
            continue
        bp = ax.boxplot(vals, positions=pos, widths=width * 0.85, whis=(5, 95), showfliers=False,
                        patch_artist=True, medianprops={'color': INK, 'linewidth': 1.2})
        for box in bp['boxes']:
            box.set(facecolor=SCHEME_COLOR[scheme], edgecolor=INK, linewidth=0.6)
    ax.set_xticks(range(len(KS)))
    ax.set_xticklabels([f'K = {K}' for K in KS])
    ax.set_title(f'Per-sample {metric} (pooled over seeds)')


if test_values:
    fig, axes = plt.subplots(1, len(EVAL_METRICS), figsize=(7 * len(EVAL_METRICS), 3.8), constrained_layout=True)
    for ax, metric in zip(np.atleast_1d(axes), EVAL_METRICS):
        grouped_boxplot(ax, metric)
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=SCHEME_COLOR[s], edgecolor=INK, linewidth=0.6, label=s)
               for s in SCHEMES]
    fig.legend(handles=handles, loc='outside lower center', ncol=len(SCHEMES), fontsize=9)
    plt.show()
    plt.close(fig)

## Confusion matrices

`Confusion TP/TN/FP/FN` are summed over the test set at `evaluation.threshold`. Each seed's matrix is normalised to fractions of the test pixels, then averaged over seeds; the cell text is mean +- std of the percentage.

In [ ]:
def confusion(run):
    vals = [last_value(run, f'Confusion {k}', {'subset': 'test'}) for k in ('TN', 'FP', 'FN', 'TP')]
    return None if any(np.isnan(v) for v in vals) else np.array(vals, dtype=float).reshape(2, 2)


confusions = {key: confusion(run) for key, run in test_runs.items()}


def cell_confusions(cell):
    cms = [confusions.get((cell[0], cell[1], seed)) for seed in sorted(test_cells.get(cell, {}))]
    return [cm / cm.sum() for cm in cms if cm is not None]


for K in KS:
    display(Markdown(f'### K = {K}'))
    fig, axes = plt.subplots(1, len(SCHEMES), figsize=(4.2 * len(SCHEMES), 3.8), constrained_layout=True, squeeze=False)
    for ax, scheme in zip(axes[0], SCHEMES):
        fr = cell_confusions((scheme, K))
        if not fr:
            mark_missing(ax, scheme, 'not run' if (scheme, K) not in test_cells else 'not tracked')
            continue
        mean, std = np.mean(fr, axis=0), np.std(fr, axis=0, ddof=1) if len(fr) > 1 else np.zeros((2, 2))
        (tn, fp), (fn, tp) = mean
        ax.imshow(mean, cmap='Blues')
        ax.grid(False)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['pred 0', 'pred 1'])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['true 0', 'true 1'])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{100 * mean[i, j]:.3f}%\n+- {100 * std[i, j]:.3f}', ha='center', va='center',
                        fontsize=9, color='white' if mean[i, j] > mean.max() / 2 else INK)
        prec = tp / (tp + fp) if tp + fp else np.nan
        rec = tp / (tp + fn) if tp + fn else np.nan
        ax.set_title(f'{scheme} ({len(fr)} seeds)\nacc {tp + tn:.4f} - prec {prec:.4f} - rec {rec:.4f}')
    plt.show()
    plt.close(fig)

## Final aggregate results

One row per (K, scheme, seed), then mean and sample std over seeds. `Recall` is TP / (TP + FN) from the confusion counts, `train loss` is the mean client train loss over the last `SMOOTH` rounds.

In [ ]:
seed_rows = []
for (scheme, K, seed), run in test_runs.items():
    c = curves.get((scheme, K, seed)) or {}
    cm = confusions.get((scheme, K, seed))
    vals = test_values.get((scheme, K, seed), {})
    seed_rows.append({
        'K': K, 'scheme': scheme, 'seed': seed,
        'ROC-AUC': c.get('roc_auc', np.nan),
        'PR-AUC': c.get('pr_auc', np.nan),
        'TPR @ FPR=5%': c.get('tpr@fpr5', np.nan),
        'Accuracy': last_value(run, 'Test Accuracy', {'subset': 'test'}),
        'Precision': last_value(run, 'Test Precision', {'subset': 'test'}),
        'Recall': cm[1, 1] / cm[1].sum() if cm is not None and cm[1].sum() else np.nan,
        'Avg NRMS': float(vals['NRMS'].mean()) if vals.get('NRMS', np.array([])).size else np.nan,
        'Avg SSIM': float(vals['SSIM'].mean()) if vals.get('SSIM', np.array([])).size else np.nan,
        f'train loss (last {SMOOTH})': train_final.get((scheme, K, seed), np.nan),
    })

seed_df = pd.DataFrame(seed_rows)
if seed_df.empty:
    final_mean = final_std = pd.DataFrame()
else:
    seed_df['scheme'] = pd.Categorical(seed_df['scheme'], SCHEMES, ordered=True)
    seed_df = seed_df.sort_values(['K', 'scheme', 'seed']).set_index(['K', 'scheme', 'seed'])
    display(Markdown('**Per seed**'))
    display(seed_df.round(4))

    grouped = seed_df.groupby(['K', 'scheme'], observed=True)
    final_mean, final_std = grouped.mean(), grouped.std(ddof=1)
    n_seeds = grouped.size().rename('seeds')
    display(Markdown('**Mean over seeds**'))
    display(final_mean.join(n_seeds).round(4))
    display(Markdown('**Std over seeds**'))
    display(final_std.round(4))
    display(Markdown('**mean +- std**'))
    display((final_mean.round(4).astype(str) + ' +- ' + final_std.round(4).astype(str)).join(n_seeds))

In [ ]:
if not final_mean.empty:
    for metric in ('ROC-AUC', 'PR-AUC', 'TPR @ FPR=5%'):
        pivot = final_mean[metric].unstack('K').reindex(SCHEMES).dropna(how='all')
        display(Markdown(f'**{metric}** - mean over seeds (rows: scheme, columns: K)'))
        display(pivot.round(4))
        if 'iid' in pivot.index:
            display(Markdown(f'**{metric} gap vs iid at the same K** (mean over seeds)'))
            display(pivot.sub(pivot.loc['iid'], axis=1).drop(index='iid').round(4))

In [ ]:
# Headline cross-size comparison: mean +- std over seeds, dots = single seeds.
# Dashed: centralised RouteNet baseline (TCAD 2023), not a run of this sweep.
BASELINE = {'ROC-AUC': 0.95, 'PR-AUC': 0.63}

if not final_mean.empty:
    panels = ['ROC-AUC', 'PR-AUC', 'TPR @ FPR=5%']
    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 3.8), constrained_layout=True)
    for ax, metric in zip(axes, panels):
        for scheme in SCHEMES:
            if scheme not in final_mean.index.get_level_values('scheme'):
                continue
            mean = final_mean[metric].xs(scheme, level='scheme').dropna()
            std = final_std[metric].xs(scheme, level='scheme').reindex(mean.index).fillna(0)
            ax.errorbar(mean.index, mean.values, yerr=std.values, color=SCHEME_COLOR[scheme],
                        marker=SCHEME_MARKER[scheme], capsize=3, label=scheme)
            pts = seed_df[metric].xs(scheme, level='scheme').dropna()
            ax.scatter(pts.index.get_level_values('K'), pts.values, color=SCHEME_COLOR[scheme], s=10, alpha=0.4)
        if metric in BASELINE:
            ax.axhline(BASELINE[metric], color=MUTED, linestyle='--', linewidth=1)
            ax.text(KS[-1], BASELINE[metric], 'centralised RouteNet ', ha='right', va='bottom', fontsize=7,
                    color=INK, bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1})
        ax.set_title(metric)
        log_k_axis(ax)
    scheme_legend(axes[0])
    fig.suptitle('Test performance across participation sizes (mean +- std over seeds)')
    plt.show()
    plt.close(fig)